# 장기 기억: 엔티티와 관계

`Entity`와 엔티티 간 관계를 직접 저장하고 Neo4j 그래프로 확인한다.
사람과 장소를 각각 노드로 표현하고, 두 노드 사이에 `BASED_IN` 의미를 갖는 관계를 저장한다.

실행 흐름은 클라이언트 연결, 엔티티·관계 저장, 시각화, 데이터 삭제, 연결 종료 순서이다.
사실의 신뢰도와 유효기간은 [Fact 실험](03_long_term_memory_fact.ipynb)에서 별도로 다룬다.

## 실행 준비

로그 출력을 설정하고 공통 설정 모듈에서 `MemorySettings`를 구성한다.
클라이언트를 한 번 연결한 뒤 저장·시각화·삭제 단계에서 재사용한다. 각 노트북은 별도의 커널에서 독립적으로 실행할 수 있다.

In [ ]:
import logging

logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)
logging.getLogger("transformers.configuration_utils").setLevel(logging.ERROR)

In [ ]:
from config.memory_client import load_memory_settings, print_memory_settings
from neo4j_agent_memory import MemorySettings

settings: MemorySettings = load_memory_settings()
print_memory_settings(settings)

In [ ]:
from neo4j_agent_memory import MemoryClient

memory_client = MemoryClient(settings)

try:
    await memory_client.connect()
except:
    await memory_client.close()
    raise

print("✓ Memory client opened")

## 엔티티와 관계 저장

`add_entity()`로 사람과 장소를 저장하고, 반환된 엔티티 ID를 `add_relationship()`에 전달하여 관계를 생성한다.
기존 엔티티가 재사용될 수 있으므로 후속 조회와 삭제에는 반환 객체의 ID를 사용한다.
이 실험은 대화를 생성하지 않으며 `session_id`를 사용하지 않는다.

In [ ]:
# 1. 사람 엔티티
jeeklee, _ = await memory_client.long_term.add_entity(
    name="jeeklee",
    entity_type="PERSON",
    subtype="employee",
    description= "jeeklee는 FDE(Forward Deployed Engineer)로, 고객사 현장에 투입되어 제품 도입·연동·커스터마이징을 담당한다.",
    attributes={
        "role": "Forward Deployed Engineer",
        "role_abbr": "FDE",
    },
    generate_embedding=True,
)

# 2. 장소 엔티티
seongsu, _ = await memory_client.long_term.add_entity(
    name="성수",
    entity_type="LOCATION",
    subtype="dong",
    description="성수(성수동)는 서울특별시 성동구에 있는 지역이다. 수제화 공방과 공장 지대였던 곳이 카페, 팝업스토어, 스타트업 사무실이 모인 상권으로 바뀌었다.",
    attributes={
        "aliases": ["성수동", "Seongsu", "Seongsu-dong"],
        "country": "대한민국",
        "city": "서울특별시",
        "district": "성동구",
    },
    generate_embedding=True,
)

# 3. 관계: jeeklee --BASED_IN--> 성수
jeeklee_based_in_seongsu = await memory_client.long_term.add_relationship(
    source=jeeklee.id,
    target=seongsu.id,
    relationship_type="BASED_IN",
    description="jeeklee는 성수를 거점으로 활동한다.",
    attributes={
        "since": "2023-03",         # 알면 "2025-03" 형식으로
        "confidence": 1.0,     # 사용자가 직접 입력한 사실이므로 1.0
    },
)

## 엔티티와 관계 시각화

직접 저장한 두 엔티티의 ID와 관계 유형으로 Neo4j 그래프를 조회한다. 이 엔티티들은 대화와 별도로 저장되므로 `Conversation`이나 `session_id`를 조회 기준으로 사용하지 않는다.

물리적인 관계 타입은 `RELATED_TO`이며, `BASED_IN`과 같은 의미적 관계 유형은 `r.type` 속성에 저장된다.
공유 모듈 `graph_visualizer`의 `visualize()`에 기존 세션과 쿼리 파라미터를 전달하여 엔티티 이름과 관계 유형을 표시한다.
세션은 조회 후 닫히고 메모리 클라이언트는 이후 실험에서 재사용한다.


In [ ]:
from typing import LiteralString

from graph_visualizer import visualize

graph_query: LiteralString = """
MATCH path = (source:Entity {id: $source_id})
    -[r:RELATED_TO]->(target:Entity {id: $target_id})
WHERE r.type = $relationship_type
RETURN path
"""

async with memory_client.graph.session() as graph_session:
    await visualize(
        graph_session,
        graph_query,
        {
            "source_id": str(jeeklee.id),
            "target_id": str(seongsu.id),
            "relationship_type": jeeklee_based_in_seongsu.type,
        },
    )


## 실험 데이터 삭제

`jeeklee_based_in_seongsu`의 ID와 두 엔티티의 ID를 기준으로 실험 관계를 삭제한다.
관계 삭제 후 `jeeklee`, `seongsu` 중 다른 노드와 연결되지 않은 엔티티를 삭제한다. 다른 연결이 남아 있는 엔티티는 보존한다.

삭제는 하나의 쿼리로 처리되어 오류 발생 시 함께 롤백된다. 데이터가 기존 엔티티·관계와 병합된 경우에도 같은 ID의 항목이 삭제 대상이 된다.
데이터를 보존하는 경우 이 셀을 생략하고 클라이언트 종료 셀만 실행한다.

In [ ]:
from typing import LiteralString

cleanup_query: LiteralString = """
CALL () {
    MATCH (:Entity {id: $source_id})-[r:RELATED_TO]->(:Entity {id: $target_id})
    WHERE r.id = $relationship_id
    DELETE r
    RETURN count(*) AS deleted_relationships
}
CALL () {
    MATCH (e:Entity)
    WHERE e.id IN $entity_ids AND NOT (e)--()
    DELETE e
    RETURN count(*) AS deleted_entities
}
CALL () {
    MATCH (e:Entity)
    WHERE e.id IN $entity_ids
    RETURN count(*) AS retained_entities
}
RETURN deleted_relationships, deleted_entities, retained_entities
"""

async with memory_client.graph.session() as graph_session:
    result = await graph_session.run(
        cleanup_query,
        {
            "source_id": str(jeeklee.id),
            "target_id": str(seongsu.id),
            "relationship_id": str(jeeklee_based_in_seongsu.id),
            "entity_ids": [str(jeeklee.id), str(seongsu.id)],
        },
    )
    deleted = await result.single(strict=True)
    await result.consume()

print(f"✓ Deleted relationships: {deleted['deleted_relationships']}")
print(f"✓ Deleted entities: {deleted['deleted_entities']}")
print(f"  Retained entities with other connections: {deleted['retained_entities']}")

## 메모리 클라이언트 종료

`close()`로 메모리 클라이언트의 연결과 백그라운드 작업을 정리한다.
연결 종료는 DB에 저장된 데이터를 삭제하지 않으므로, 데이터 삭제 여부와 관계없이 실험을 마칠 때 실행한다.
종료 후 실험을 재개하려면 클라이언트 생성 및 연결 셀부터 다시 실행한다.

In [ ]:
await memory_client.close()
print("✓ Memory client closed")